In [1]:
import cv2
import numpy as np
import time
import queue
from typing import List, Optional
from tensorflow.keras.models import Model, load_model
from landmarkers.inferences import InferenceSequence, Inference
from landmarkers.mp.hands import MediapipeHandsMetadata, MPLiveStreamLandmarker
from landmarkers.visualization.layers import SequenceLayer, PointsLayer, BBoxLayer
from landmarkers.visualization import Viewer, ViewerBuilder
from landmarkers.visualization.visualizers import LandmarksSequenceVisualizer

# -------------------------
# Configuración
# -------------------------
ACTIONS: np.ndarray = np.array(["jump", "shoot", "none"])
SEQUENCE_LENGTH: int = 15
MODEL_EXPORT_NAME: str = "hand_gesture_model.h5"
THRESHOLD: float = 0.85
PRED_BUFFER_SIZE: int = 5
NUM_LANDMARKS: int = 21  # Cantidad de landmarks de la mano

# -------------------------
# Carga de modelos y viewer
# -------------------------
model: Model = load_model(MODEL_EXPORT_NAME)
inference_sequence: InferenceSequence = InferenceSequence(
    fixed_buffer_length=SEQUENCE_LENGTH
)

landmark_viewer: Viewer = (
    ViewerBuilder()
    .add_layer(SequenceLayer([PointsLayer(), BBoxLayer()], step=5, time_fade=True))
    .build()
)

# -------------------------
# Funciones auxiliares
# -------------------------
def get_timestamp_ms() -> int:
    return int(time.time() * 1000)


def preprocess_landmarks(landmarks_seq) -> np.ndarray:
    resampled = landmarks_seq.resample()
    centered = resampled.centered(0).array
    return centered.reshape(SEQUENCE_LENGTH, -1)


def predict_gesture(
    features: np.ndarray,
    model: Model,
    pred_buffer: List[np.ndarray],
    threshold: float = THRESHOLD,
) -> Optional[str]:
    res: np.ndarray = model.predict(np.expand_dims(features, axis=0), verbose=False)[0]
    pred_buffer.append(res)
    avg: np.ndarray = np.mean(pred_buffer[-PRED_BUFFER_SIZE:], axis=0)
    idx: int = int(np.argmax(avg))
    if avg[idx] > threshold and ACTIONS[idx] != "none":
        return str(ACTIONS[idx])
    return None


def render_frame(
    frame: np.ndarray,
    inference_sequence: InferenceSequence,
    landmark_viewer: Viewer,
    gesture: Optional[str] = None,
) -> np.ndarray:
    if gesture:
        cv2.putText(
            frame, gesture, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3
        )
    return frame


def get_right_hand_inference(
    inferences: Optional[List[Inference]],
) -> Optional[Inference]:
    if inferences is None:
        return None
    right_hand: List[Inference] = [
        inf for inf in inferences if inf.metadata.category_name == "Right"
    ]
    return right_hand[0] if right_hand else None


def create_empty_right_hand_inference() -> Inference:
    zeros = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
    metadata: MediapipeHandsMetadata = MediapipeHandsMetadata(
        category_name="Right", index=0, score=0.0
    )
    return Inference(landmarks=zeros, world_landmarks=zeros, metadata=metadata)


# -------------------------
# Cola para comunicarse entre hilo de MediaPipe y hilo principal
# -------------------------
frame_queue: "queue.Queue[tuple[np.ndarray, List[Inference], int]]" = queue.Queue(maxsize=1)
pred_buffer: List[np.ndarray] = []

def stream_callback(
    inferences: List[Inference], frame: np.ndarray, timestamp_ms: int
) -> None:
    """Callback que solo pone datos en la cola para el hilo principal."""
    try:
        frame_queue.put_nowait((frame, inferences, timestamp_ms))
    except queue.Full:
        pass  # descartamos frame si no hay espacio, evitando bloqueos


# -------------------------
# Main loop con MPLiveStreamLandmarker
# -------------------------
cap: cv2.VideoCapture = cv2.VideoCapture(0)

with MPLiveStreamLandmarker(
    model_path="hand_landmarker.task",
    callback=stream_callback,
    num_hands=2,
) as live_landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        timestamp = get_timestamp_ms()
        live_landmarker.infer(frame, timestamp)

        # Procesar el último frame disponible en el hilo principal
        if not frame_queue.empty():
            f, inferences, ts = frame_queue.get()
            right_hand: Inference = get_right_hand_inference(inferences) or create_empty_right_hand_inference()
            inference_sequence.append(right_hand, ts)

            gesture: Optional[str] = None
            if len(inference_sequence) == SEQUENCE_LENGTH:
                features = preprocess_landmarks(inference_sequence.landmarks_sequence)
                gesture = predict_gesture(features, model, pred_buffer)

            show_frame = render_frame(f, inference_sequence, landmark_viewer, gesture)
            cv2.imshow("Feed", show_frame)

        if cv2.waitKey(10) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()


2026-01-29 00:34:48.213493: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-29 00:34:49.141020: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-29 00:34:51.363365: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
2026-01-29 00:34:53.429886: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA erro